In [1]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import CRS
import subprocess
import time
from tqdm import tqdm

# Configurations

In [2]:
# API endpoint
base_url = "https://naptan.api.dft.gov.uk/v1/StopPoints"

# PostGIS config
ogr2ogr_path = r"C:\Program Files\QGIS 3.34.11\bin\ogr2ogr.exe"
gpkg_path = r"N:\Geodatabase\Raw_Data\GPKG_temp_folder_for_geodatabse_upload/temp_upload.gpkg"
target_crs = CRS.from_epsg(27700)
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
pg_conn_string = f"PG:host={db_host} dbname={db_name} port={db_port}"
layer_name = "mob_naptan_stops"

# Fetch all active stops from the NaPTAN API

In [3]:
def fetch_all_active_stops():
    all_data = []
    page = 1
    page_size = 1000

    while True:
        params = {"status": "active", "page": page, "pageSize": page_size}
        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            raise Exception(f"Request failed on page {page}")

        page_data = response.json()
        items = page_data.get("value", [])
        all_data.extend(items)

        if not page_data.get("nextPage"):
            break
        page += 1

    return pd.DataFrame(all_data)

df = fetch_all_active_stops()
df.columns = [c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "") for c in df.columns]
print(f"Fetched {len(df)} active stops")

Exception: Request failed on page 1

# Convert to GeoDataFrame and reproject

In [21]:
df = df.dropna(subset=["latitude", "longitude"])
geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]

gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf = gdf.to_crs(target_crs)

# Save to GeoPackage
gdf.to_file(gpkg_path, driver="GPKG", layer=layer_name)
print("GeoPackage saved:", gpkg_path)

# Upload to PostGIS using ogr2ogr

In [22]:
print("Uploading to PostGIS...")
start_time = time.time()

ogr2ogr_cmd = [
    ogr2ogr_path,
    "-f", "PostgreSQL",
    pg_conn_string,
    gpkg_path,
    "-nln", f"{db_schema}.{layer_name}",
    "-nlt", "PROMOTE_TO_MULTI",
    "-lco", "GEOMETRY_NAME=geometry",
    "-lco", "SPATIAL_INDEX=GIST",
    "-lco", "SCHEMA=" + db_schema,
    "-lco", "OVERWRITE=YES",
    "-overwrite"
]

with tqdm(total=100, desc=f"Uploading {layer_name}", bar_format="{l_bar}{bar}| {elapsed}") as pbar:
    result = subprocess.run(ogr2ogr_cmd, capture_output=True, text=True)
    pbar.update(100)

if result.returncode != 0:
    print("❌ Upload failed:")
    print(result.stderr)
else:
    print("✅ Upload successful.")
    print(f"⏱ Time taken: {time.time() - start_time:.2f} seconds")

Reading input layer...
Detected input CRS: 4326
Target CRS: 27700
Reprojecting to target CRS...
Converting .geojson to GeoPackage for upload...


C:\Users\abhimanya.achara\Anaconda3\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


Uploading to PostGIS via ogr2ogr...


Uploading: 100%|██████████| 24:24


 Upload completed successfully.
 Time taken: 1464.76 seconds
